# CatBoost and TabPFN in QBioCode

Two classical learners were added alongside XGBoost, and neither behaves quite like the
scikit-learn estimators around them. This notebook runs both, and spends most of its time
on the three places where that difference is visible to you:

1. **CatBoost beside XGBoost and a random forest** — the head-to-head, tuned and untuned.
2. **A hyperparameter conflict that depends on configuration rather than on your data** —
   the one genuine trap in the CatBoost surface, and what QBioCode says when you hit it.
3. **A hyperparameter that silently does nothing**, measured rather than asserted.
4. **TabPFN**, a pretrained transformer that classifies without training.

## About TabPFN before you start

TabPFN needs only the optional extra — **no API key, no license acceptance, no GPU**:

```bash
pip install "qbiocode[tabpfn]"
```

QBioCode pins `model_version="v2"`, whose weights are published under the Prior Labs
License (Apache 2.0 plus an attribution clause) and download anonymously on first fit.

Two things commonly assumed and both wrong:

- **A GPU is not required.** TabPFN runs on CPU. This notebook pins `device="cpu"`
  deliberately, because MPS and CUDA are not numerically identical to CPU and a benchmark
  that silently changes accelerator is not reproducible across machines.
- **An API key is not required** — for `v2`. It is only needed for the newer checkpoints,
  and those come with strings attached (see below).

### Why v2 and not v3

TabPFN's *code* is Apache 2.0 plus attribution. Its *weights* are licensed per version:

| `model_version` | Weights license | Commercial use |
| --- | --- | --- |
| `v2` (QBioCode default) | Prior Labs License v1.1 (Apache 2.0 + attribution) | **Permitted** |
| `v2.5` / `v2.6` / `v3` | Per-version Non-Commercial License | No |

The newer three are **non-production as well as non-commercial**, and reaching them needs
an interactive license acceptance against a Prior Labs account — so they cannot be fetched
unattended either. Upstream's constructor defaults to `v3`; QBioCode does not, because it
is Apache-2.0 software whose users include companies. Selecting a restricted version warns
and names the license.

CatBoost has no such considerations — it is a core dependency, always available.

<a id="setup"></a>
## 1. Setup

In [1]:
# On macOS a QBioCode process ends up with several LLVM OpenMP runtimes mapped in under
# one install name (torch, qiskit-aer, xgboost). Whichever initialises second dies in
# __kmp_fork_barrier the first time it opens a parallel region -- below Python, so there
# is no traceback and a notebook front end can only report "kernel died".
# This must run BEFORE qbiocode/torch are imported, so keep it in the first cell.
# (CatBoost is unaffected: it uses its own thread pool rather than OpenMP.)
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")

import pathlib
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")

import qbiocode
from qbiocode import scale_train_test
from qbiocode.evaluation.model_run import model_run

# Which QBioCode answered? Printed relative to home -- the absolute install path would be
# committed into a published page. If this is not the checkout you are editing, stop and
# change kernel: every number below would describe different code.
loaded = pathlib.Path(qbiocode.__file__).parent.parent
try:
    shown = "~/" + str(loaded.relative_to(pathlib.Path.home()))
except ValueError:
    shown = loaded.name
print(f"qbiocode {qbiocode.__version__} from {shown}")

qbiocode 0.2.0 from ~/Documents/Q/qbc/82826/fork/QBioCode


A small, learnable, binary problem with one interaction term, so the tree ensembles have
something a linear model would miss. Scaled *after* splitting: fitting a scaler on the
full matrix leaks test-set statistics into training and flatters every number below.

In [2]:
rng = np.random.default_rng(0)
N_SAMPLES, N_FEATURES, SPLIT = 160, 12, 110

X = rng.normal(size=(N_SAMPLES, N_FEATURES))
signal = X[:, 0] + 0.7 * X[:, 1] - 0.5 * X[:, 2] * X[:, 3]   # the interaction
y = (signal + 0.5 * rng.normal(size=N_SAMPLES) > 0).astype(int)

X_train, X_test = X[:SPLIT], X[SPLIT:]
y_train, y_test = y[:SPLIT], y[SPLIT:]
X_train, X_test = scale_train_test(X_train, X_test)

print(f"train {X_train.shape}   test {X_test.shape}")
print(f"class balance: train {y_train.mean():.2f}, test {y_test.mean():.2f}")

train (110, 12)   test (50, 12)
class balance: train 0.46, test 0.50


<a id="boosters"></a>
## 2. CatBoost beside XGBoost

Both are selected exactly like any other model: a name in `model` and an optional
`<name>_args` block. Nothing about the dispatch is special-cased.

In [3]:
def run(models, label, **overrides):
    """One model_run, flattened into a tidy frame.

    `model_run` returns a dict keyed `results_<model>`, where the label is the model's
    display name rather than the dispatch key -- and gains an `_opt` suffix when tuning is
    on. So the columns are found rather than assumed.
    """
    args = {
        "model": models, "grid_search": False, "seed": 42,
        "n_jobs": len(models), "average": "weighted",
        **overrides,
    }
    raw = model_run(X_train, X_test, y_train, y_test, "demo", args)

    rows, tuned = [], {}
    for key, value in raw.items():
        if not key.startswith("results_"):
            continue
        metrics = value[0]
        rows.append({
            "run": label, "model": metrics["model"],
            "accuracy": metrics["accuracy"], "f1": metrics["f1_score"],
            "auc": metrics["auc"], "seconds": round(metrics["time"], 2),
        })
        if "BestParams_Tuned" in metrics:
            tuned[metrics["model"]] = metrics["BestParams_Tuned"]
    frame = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
    return (frame, tuned) if tuned else frame


DEFAULTS = {
    "catboost_args": {"iterations": 200, "learning_rate": 0.1, "depth": 4},
    "xgb_args": {"n_estimators": 200, "learning_rate": 0.1, "max_depth": 4},
}

untuned = run(["catboost", "xgb", "rf"], "defaults", **DEFAULTS)
untuned

,run,model,accuracy,f1,auc,seconds
0,defaults,xgb,0.82,0.819928,0.82,0.03
1,defaults,catboost,0.78,0.779912,0.78,0.12
2,defaults,rf,0.78,0.779205,0.78,0.06


### One thing that had to be handled for you

CatBoost writes a `catboost_info/` directory of training logs into the **current working
directory** on every single fit. QProfiler fans models out over joblib workers that share
a CWD, so that is both a race and litter left in your project. QBioCode passes
`allow_writing_files=False` to every CatBoost estimator it builds, so:

In [4]:
print("in the working directory now:",
      sorted(p.name for p in pathlib.Path(".").iterdir()) or ["nothing"])

in the working directory now: ['catboost_and_tabpfn.ipynb']


<a id="tuning"></a>
## 3. Tuning CatBoost

The same `gridsearch_<model>_args` block and the same two engines as every other classical
model. A list is a categorical choice; a `{low, high}` mapping is a continuous range that
only the Optuna engine can sample.

In [5]:
TUNED_BLOCK = {
    "gridsearch_catboost_args": {
        "iterations": [100, 200, 400],
        "learning_rate": {"low": 1.0e-2, "high": 3.0e-1, "log": True},
        "depth": [3, 4, 6],
        "l2_leaf_reg": {"low": 1.0, "high": 10.0},
    },
}

tuned, best = run(["catboost"], "tuned", grid_search=True, tuner="optuna",
                  n_trials=12, cross_validation=3, **TUNED_BLOCK)
display(pd.concat([untuned, tuned], ignore_index=True))
print("best hyperparameters found:")
for name, value in best["catboost_opt"].items():
    print(f"  {name:16} {value}")

,run,model,accuracy,f1,auc,seconds
0,defaults,xgb,0.82,0.819928,0.82,0.03
1,defaults,catboost,0.78,0.779912,0.78,0.12
2,defaults,rf,0.78,0.779205,0.78,0.06
3,tuned,catboost_opt,0.82,0.819928,0.82,2.27


best hyperparameters found:
  iterations       200
  learning_rate    0.03023795012558475
  depth            6
  l2_leaf_reg      4.210779940242303


`learning_rate` and `l2_leaf_reg` came back as values no list of decades contains — that
is what the range syntax buys. Under `tuner: "grid"` the same block is refused, by name,
because a grid can only enumerate.

<a id="bootstrap"></a>
## 4. The one real trap: two hyperparameters that cannot coexist

`subsample` and `bagging_temperature` belong to **mutually exclusive** CatBoost bootstrap
schemes. `subsample` needs Bernoulli/MVS/Poisson; `bagging_temperature` needs Bayesian.
Left unpinned, CatBoost chooses the scheme from the loss — so whether your config is valid
depends on something you did not write in it.

QBioCode pins the scheme as soon as you name either one, and refuses a genuinely
contradictory block *before* the search starts rather than partway through it:

In [6]:
from qbiocode.learning.compute_catboost import compute_catboost, compute_catboost_opt

ARGS = {"grid_search": False}
CASES = [
    ("subsample alone",           {"subsample": 0.8}),
    ("bagging_temperature alone", {"bagging_temperature": 0.5}),
    ("both at once",              {"subsample": 0.8, "bagging_temperature": 0.5}),
    ("subsample + Bayesian",      {"subsample": 0.8, "bootstrap_type": "Bayesian"}),
]

for label, kwargs in CASES:
    try:
        frame = compute_catboost(X_train, X_test, y_train, y_test, ARGS,
                                 iterations=50, random_state=42, **kwargs)
        metrics = [v for v in frame["results_catboost"] if isinstance(v, dict)][0]
        scheme = metrics["Model_Parameters"].get("bootstrap_type", "(CatBoost default)")
        print(f"{label:26} -> fitted, bootstrap_type pinned to {scheme!r}")
    except ValueError as error:
        print(f"{label:26} -> refused:")
        print(f"{'':30}{str(error).splitlines()[0][:88]}")

subsample alone            -> fitted, bootstrap_type pinned to 'Bernoulli'
bagging_temperature alone  -> fitted, bootstrap_type pinned to 'Bayesian'
both at once               -> refused:
                              'catboost_args' names both 'subsample' and 'bagging_temperature', which belong to mutual
subsample + Bayesian       -> refused:
                              'catboost_args' sets 'subsample', which the Bayesian bootstrap does not support, but 'bo


### Why this is not a multiclass-only concern

QProfiler targets binary classification, and CatBoost infers `Logloss` for a two-class
target — which defaults to the `MVS` bootstrap, where `subsample` is fine. So it is
tempting to conclude the conflict is unreachable here.

It is reachable, through `loss_function`. `"MultiClass"` is legal on a two-class target,
and setting it moves CatBoost onto the Bayesian bootstrap *and* changes the shape of
`predict()`:

In [7]:
from catboost import CatBoostClassifier

rows = []
for loss in (None, "Logloss", "MultiClass"):
    extra = {} if loss is None else {"loss_function": loss}
    model = CatBoostClassifier(iterations=20, random_state=42, verbose=False,
                               allow_writing_files=False, **extra).fit(X_train, y_train)
    rows.append({
        "loss_function": str(loss),
        "default bootstrap": model.get_all_params()["bootstrap_type"],
        "predict() shape": str(model.predict(X_test).shape),
    })
pd.DataFrame(rows)

,loss_function,default bootstrap,predict() shape
0,None,MVS,"(50,)"
1,Logloss,MVS,"(50,)"
2,MultiClass,Bayesian,"(50, 1)"


The `(n, 1)` column in the last row is the second half of the same problem. scikit-learn
happens to squeeze it when scoring, so the metrics are right either way — but QBioCode
flattens it anyway, so that the `y_predicted_*` column in your results frame has the same
shape as every other model's rather than resting on undocumented squeezing behaviour.

<a id="inert"></a>
## 5. A hyperparameter that does nothing

CatBoost accepts `min_data_in_leaf` under every grow policy and *honours* it under only
two. At the default `SymmetricTree` it is silently inert, so searching it multiplies your
fits while every value returns the same model. Measured rather than taken on faith:

In [8]:
def probabilities(**kwargs):
    base = dict(iterations=40, random_state=42, verbose=False, allow_writing_files=False)
    fitted = CatBoostClassifier(**{**base, **kwargs}).fit(X_train, y_train)
    return fitted.predict_proba(X_test)[:, 1]


for policy in ("SymmetricTree", "Depthwise", "Lossguide"):
    identical = np.allclose(probabilities(grow_policy=policy, min_data_in_leaf=1),
                            probabilities(grow_policy=policy, min_data_in_leaf=40))
    verdict = "identical -> INERT" if identical else "different -> honoured"
    print(f"grow_policy={policy:14} min_data_in_leaf 1 vs 40: {verdict}")

grow_policy=SymmetricTree  min_data_in_leaf 1 vs 40: identical -> INERT
grow_policy=Depthwise      min_data_in_leaf 1 vs 40: different -> honoured
grow_policy=Lossguide      min_data_in_leaf 1 vs 40: different -> honoured


So QBioCode does not let you search it. Giving `min_data_in_leaf` several values is
**refused** rather than warned about — a warning would let the wasted fits happen anyway:

In [9]:
try:
    compute_catboost_opt(
        X_train, X_test, y_train, y_test, {"grid_search": True},
        cv=2, n_trials=2, iterations=[20], min_data_in_leaf=[1, 40], random_state=42,
    )
except ValueError as error:
    print("refused:\n")
    import textwrap
    print(textwrap.fill(str(error), 88, initial_indent="   ", subsequent_indent="   "))

# It is still usable as a single value, paired with a policy that honours it -- passed to
# every trial rather than searched, so it costs nothing and is not reported as "tuned".
frame = compute_catboost_opt(
    X_train, X_test, y_train, y_test, {"grid_search": True},
    cv=2, n_trials=2, iterations=[20], min_data_in_leaf=5,
    grow_policy=["Depthwise"], random_state=42,
)
best = [v for v in frame["results_catboost"] if isinstance(v, dict)][0]["BestParams_Tuned"]
print(f"\naccepted as a fixed value; tuned result reports only {sorted(best)}")

refused:

   'gridsearch_catboost_args' gives 'min_data_in_leaf' several values ([1, 40]), but it
   is not searched: CatBoost honours it only under grow_policy 'Depthwise' or
   'Lossguide', so at the default 'SymmetricTree' every value produces an identical
   model and the extra fits buy nothing. Give it a single value instead, and set
   'grow_policy' if you want it to take effect -- or search 'grow_policy' and leave this
   out.



accepted as a fixed value; tuned result reports only ['grow_policy', 'iterations']


<a id="tabpfn"></a>
## 6. TabPFN: classification without training

TabPFN is not a learner in the sense everything above is. Its weights are **frozen and
pretrained** on synthetic tabular problems; `fit` only memorises your training rows, and
`predict` attends over them. There is no training, and correspondingly nothing that
resembles model capacity to tune — every knob it exposes is an *inference* setting.

That is also why it needs no tuning to be competitive, which is the point of the model on
the small datasets QProfiler targets.

In [10]:
from qbiocode.learning.compute_tabpfn import (
    TABPFN_DEFAULT_VERSION,
    TABPFN_MAX_CLASSES,
    compute_tabpfn,
    resolve_model_path,
    tabpfn_is_available,
)

print(f"[tabpfn] extra installed : {tabpfn_is_available()}")
print(f"pinned model_version     : {TABPFN_DEFAULT_VERSION}  (commercially usable weights)")
print(f"class ceiling {TABPFN_MAX_CLASSES}; this target has {len(np.unique(y_train))} -- fine")
print()

# No token, no license acceptance, no accelerator. The checkpoint downloads on first use.
tabpfn_result = compute_tabpfn(
    X_train, X_test, y_train, y_test, {"grid_search": False},
    n_estimators=4, device="cpu", random_state=42,
)
metrics = [v for v in tabpfn_result["results_tabpfn"] if isinstance(v, dict)][0]
print(f"TabPFN on CPU: accuracy={metrics['accuracy']:.3f} f1={metrics['f1_score']:.3f} "
      f"in {metrics['time']:.1f}s")

comparison = pd.concat([
    untuned, tuned,
    pd.DataFrame([{"run": "pretrained", "model": "tabpfn",
                   "accuracy": metrics["accuracy"], "f1": metrics["f1_score"],
                   "auc": metrics["auc"], "seconds": round(metrics["time"], 2)}]),
], ignore_index=True).sort_values("f1", ascending=False).reset_index(drop=True)
display(comparison)
print("No training, no tuning, no hyperparameter search -- one forward pass over the rows.")

[tabpfn] extra installed : True
pinned model_version     : v2  (commercially usable weights)
class ceiling 10; this target has 2 -- fine



TabPFN on CPU: accuracy=0.840 f1=0.840 in 0.7s


,run,model,accuracy,f1,auc,seconds
0,pretrained,tabpfn,0.84,0.840000,0.84,0.65
1,defaults,xgb,0.82,0.819928,0.82,0.03
2,tuned,catboost_opt,0.82,0.819928,0.82,2.27
3,defaults,catboost,0.78,0.779912,0.78,0.12
4,defaults,rf,0.78,0.779205,0.78,0.06


No training, no tuning, no hyperparameter search -- one forward pass over the rows.


<a id="takeaways"></a>
## Takeaways

- **CatBoost is a drop-in classical learner**: a name in `model`, an optional args block,
  an `_opt` twin under `grid_search: True`, both tuning engines. It is a core dependency,
  so it always works.
- **Run it beside XGBoost rather than instead of it.** Symmetric trees and ordered
  boosting behave differently on small, wide tables; on this dataset untuned XGBoost beat
  untuned CatBoost, and tuning closed the gap. Which wins is an empirical question per
  dataset, which is what QProfiler is for.
- **`subsample` and `bagging_temperature` cannot coexist**, and the default that decides
  it comes from the loss rather than from your config. QBioCode pins the scheme and
  refuses a contradictory block up front, naming the config key.
- **`min_data_in_leaf` is inert under the default grow policy**, so QBioCode refuses to
  *search* it. Pair a single value with `grow_policy: Depthwise` to make it bite, or leave
  it out.
- **TabPFN needs neither a GPU nor an API key** at the pinned `v2`. It runs on CPU, its
  ceiling is 10 classes, and it is competitive here with no training and no tuning at all.
- **The model version is a licensing decision.** `v2`'s weights permit commercial use;
  `v2.5`/`v2.6`/`v3` are non-commercial *and* non-production and need an interactive
  license acceptance. QBioCode pins `v2` and warns if you opt out of it.
- **If you do opt into a restricted version**, keep its API key out of the repository:
  `~/.config/qbiocode/tabpfn.json` (mode 0600) is read automatically by QProfiler, and by
  `qbiocode.utils.load_tabpfn_token()` elsewhere. It lives outside the checkout so that
  committing it is impossible rather than merely discouraged.

### Where to go next

- `tutorial/Hyperparameter_Tuning/optuna_vs_gridsearch.ipynb` — the two search engines
  timed against each other, and quantum tuning.
- `tutorial/QProfiler/example_qprofiler.ipynb` — the full benchmark sweep, which now
  includes `catboost` in its model list.
- `docs/source/apps/config.md` — every hyperparameter both learners accept.